# Load a pretrained decoder and keep training it

You will load a Hugging Face decoder checkpoint into the same `CausalTransformer` the previous notebooks trained from scratch, generate text from it, continue its pretraining for a few steps with the regular `LMObjective`, and export the result back into the Hugging Face layout, where the loader reads it again. No weights are renamed by hand anywhere: the decoder's parameter tree already carries the Hugging Face names, so a load or a save is a translation of names, transposes and config fields.

Two routes share every cell but the configuration:

- **Offline route (default, runs here).** A checkpoint with random weights in the Qwen3 layout, written by Dew's own exporter in the first data cell, with the byte tokenizer. It exercises the whole round trip on a CPU in a minute: load, generate, train, export, reload. The text it generates is noise, because the weights are.
- **Qwen3-0.6B route.** `CHECKPOINT = "Qwen/Qwen3-0.6B"` downloads the 1.2 GB checkpoint and its tokenizer from the Hub and continues pretraining on Tiny Shakespeare. The parameters, Adam moments and EMA copy of a 0.6B model in fp32 come to about 12 GB before activations, so it wants a 40 GB accelerator; on a 16 GB card, point it at a smaller decoder. That route was not executed while writing this notebook.

In [ ]:
# Install cell: only runs on Colab (import google.colab succeeds there).
# A TPU runtime gets jax[tpu] instead of jax[cuda12]. Locally this cell does nothing.
import os
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

if IN_COLAB:
    jax_spec = "jax[tpu]" if "COLAB_TPU_ADDR" in os.environ else "jax[cuda12]"
    %pip install -q "dew-ml[interop] @ git+https://github.com/AshishKumar4/dew" {jax_spec}

In [ ]:
CHECKPOINT = "checkpoints/random-qwen3"  # a local HF-layout directory, or a hub repo id such as "Qwen/Qwen3-0.6B"
TOKENIZER = "byte"                       # the ids the checkpoint was trained on; "Qwen/Qwen3-0.6B" for that checkpoint
DATA_DIR = "data/08-tokens"
OUT_DIR = "runs/08-exported"
STEPS = 30
BATCH_SIZE = 4
SEQUENCE_LENGTH = 64
LEARNING_RATE = 1e-4
MAX_NEW_TOKENS = 32
PROMPT = "To be, or not to be"

In [ ]:
import jax
print("devices:", jax.devices())
print("backend:", jax.default_backend())

## A checkpoint in the Hugging Face layout

The offline route needs a checkpoint to load, so it writes one: a small decoder with Qwen3's ingredients (q/k normalisation, grouped-query attention, a gated MLP) is built from the registry, initialised with random weights, and written out by `save_pretrained_decoder` as `config.json`, `model.safetensors` and `generation_config.json`. The family is read off the model: q/k norms mean `model_type` comes out as `qwen3`. A real checkpoint from the Hub has the same three files with trained weights; this cell does nothing when `CHECKPOINT` names one.

In [ ]:
import os

import jax.numpy as jnp
from dew import models
from dew.interop import save_pretrained_decoder

if CHECKPOINT.startswith("checkpoints/"):
    source = models.build(
        "causal_transformer", vocab_size=256, emb_features=64, num_layers=2,
        num_heads=4, num_kv_heads=2, head_dim=16, mlp="swiglu", mlp_features=128,
        qk_norm=True, max_seq_len=128, dtype="float32", attention_impl="xla")
    random_variables = source.init(jax.random.key(0), jnp.zeros((1, 8), jnp.int32))
    save_pretrained_decoder(source, random_variables, CHECKPOINT, tokenizer_name=TOKENIZER)
    print("wrote", CHECKPOINT, sorted(os.listdir(CHECKPOINT)))

## Load the checkpoint

`load_pretrained_decoder` takes a hub repo id or a local directory in the Hugging Face layout, translates the config into `CausalTransformer` fields and the weights onto its parameter tree, and returns the model, its variables, and the Dew config the translation produced. The parameters arrive in fp32 and the model computes in the `dtype` you ask for. A config field that changes what the model computes and has no counterpart in Dew raises an error naming the field rather than being dropped.

In [ ]:
from dew.interop import load_pretrained_decoder

model, variables, model_config = load_pretrained_decoder(
    CHECKPOINT, dtype="float32", attention_impl="xla",
    max_seq_len=SEQUENCE_LENGTH + MAX_NEW_TOKENS + 16)
n_params = sum(int(x.size) for x in jax.tree_util.tree_leaves(variables))
print(f"{n_params / 1e6:.2f}M parameters")
print({k: model_config[k] for k in
       ("vocab_size", "emb_features", "num_layers", "num_heads", "num_kv_heads",
        "head_dim", "qk_norm", "max_seq_len")})

## Generation before training

`generate` prefills the prompt into a KV cache and decodes in a compiled loop. `Sampling(temperature=0)` selects the highest-scoring token; `Generation.tokens` holds the prompt and the continuation. The tokenizer is the checkpoint's own, because the model was trained on those ids: `HFTokenizer` wraps any Hugging Face tokenizer behind the same `encode`/`decode` the byte tokenizer has.

In [ ]:
from dew.data.text import ByteTokenizer, HFTokenizer
from dew.sampling import Sampling, generate

tokenizer = ByteTokenizer() if TOKENIZER == "byte" else HFTokenizer(TOKENIZER)
prompt = jnp.asarray([tokenizer.encode(PROMPT)], jnp.int32)
before = generate(model, variables, prompt, max_new_tokens=MAX_NEW_TOKENS,
                  key=jax.random.key(0), sampling=Sampling(temperature=0.0))
print(repr(tokenizer.decode(before.tokens[0])))

## Continued pretraining

The token files are written the way notebook 05 writes them, with the checkpoint's tokenizer. The offline route uses the generated sentence corpus; the Qwen3 route downloads Tiny Shakespeare. `LMObjective` takes the loaded `variables` through its `pretrained` argument, which is where `init` starts from instead of a fresh draw, so the trainer's whole initial state, optimizer included, is built on top of the loaded weights.

In [ ]:
import json
from pathlib import Path

import numpy as np

data_dir = Path(DATA_DIR)
data_dir.mkdir(parents=True, exist_ok=True)
raw = data_dir / "corpus.txt"
if TOKENIZER == "byte":
    rng = np.random.default_rng(0)
    subjects = ["the cat", "a dog", "the bird", "my friend", "the child"]
    verbs = ["sees", "likes", "finds", "wants", "hears"]
    objects = ["the ball", "a tree", "the river", "some food", "the moon"]
    raw.write_text("".join(f"{rng.choice(subjects)} {rng.choice(verbs)} {rng.choice(objects)}.\n"
                           for _ in range(2000)), encoding="utf-8")
elif not raw.exists():
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt", raw)

ids = np.asarray(tokenizer.encode(raw.read_text(encoding="utf-8")))
val_len = int(round(len(ids) * 0.02))
vocab = tokenizer.vocab_size
dtype = np.dtype("uint8" if vocab <= 256 else "uint16" if vocab <= 65536 else "uint32")
ids[:val_len].astype(dtype).tofile(data_dir / "val.bin")
ids[val_len:].astype(dtype).tofile(data_dir / "train.bin")
meta = {"tokenizer": TOKENIZER, "vocab_size": vocab, "dtype": dtype.name,
        "train_tokens": len(ids) - val_len, "val_tokens": val_len, "eos_id": None}
(data_dir / "meta.json").write_text(json.dumps(meta))
print(meta)

In [ ]:
import optax
from dew import Checkpoints, Trainer, metrics
from dew.data import Loading, TokenWindows
from dew.objectives.lm import LMObjective, Samples

data = TokenWindows(path=DATA_DIR, seq_len=SEQUENCE_LENGTH, val_batches=2,
                    loading=Loading(workers=0, threads=1, read_buffer=2)).load(batch=BATCH_SIZE)
objective = LMObjective(
    model, SEQUENCE_LENGTH, pretrained=variables,
    samples=Samples(prompt=tokenizer.encode(PROMPT), max_new_tokens=MAX_NEW_TOKENS,
                    sampling=Sampling(temperature=0.0), decode=tokenizer.decode))
trainer = Trainer(objective, optax.adamw(LEARNING_RATE), key=jax.random.key(0),
                  checkpoints=Checkpoints("runs/08-continued"))
state = trainer.fit(data, steps=STEPS, log_every=10, eval_every=STEPS, checkpoint_every=STEPS,
                    metrics=(metrics.perplexity(),))
print("optimizer updates:", int(state.updates))

## Export back, and reload

`save_pretrained_decoder` writes the trained variables in the Hugging Face vocabulary, using the same field map as the load run backwards. Loading the export again gives back the same parameters, and the reloaded model picks the same next token as the trained one. With `transformers` and `torch` installed, the last cell also loads the export in `transformers` and compares its next-token argmax with Dew's, which is what closes the round trip between the two frameworks; without them it says so and skips.

In [ ]:
save_pretrained_decoder(model, state.params, OUT_DIR, tokenizer_name=TOKENIZER)
print("wrote", OUT_DIR, sorted(os.listdir(OUT_DIR)))

again, reloaded, _ = load_pretrained_decoder(
    OUT_DIR, dtype="float32", attention_impl="xla", max_seq_len=model_config["max_seq_len"])
identical = all(np.array_equal(np.asarray(a), np.asarray(b))
                for a, b in zip(jax.tree_util.tree_leaves(reloaded["params"]),
                                jax.tree_util.tree_leaves(state.params["params"])))
print("reloaded parameters equal the trained ones:", identical)

trained_next = generate(model, state.params, prompt, max_new_tokens=1,
                        key=jax.random.key(0), sampling=Sampling(temperature=0.0))
reloaded_next = generate(again, reloaded, prompt, max_new_tokens=1,
                         key=jax.random.key(0), sampling=Sampling(temperature=0.0))
print("next token after training:", int(trained_next.tokens[0, -1]),
      "| from the reloaded export:", int(reloaded_next.tokens[0, -1]))

In [ ]:
try:
    import torch
    from transformers import AutoModelForCausalLM
except ModuleNotFoundError as missing:
    print(f"{missing.name} is not installed on this runtime; install torch and transformers to run the reload check")
else:
    hf_model = AutoModelForCausalLM.from_pretrained(OUT_DIR, dtype=torch.float32)
    with torch.no_grad():
        hf_logits = hf_model(input_ids=torch.tensor(np.asarray(prompt))).logits[0, -1]
    hf_next = int(hf_logits.argmax())
    print("next token id from transformers:", hf_next, "| from dew:", int(trained_next.tokens[0, -1]),
          "| match:", hf_next == int(trained_next.tokens[0, -1]))

## Where to go next

The families the translator accepts, and what each one refuses, are listed in the [model family reference](../docs/reference/model-families.md). The recipe flag `--pretrained` runs this notebook's flow at scale: `python recipes/lm/train.py data:token-windows --data.path <dir> --pretrained Qwen/Qwen3-0.6B --tokenizer Qwen/Qwen3-0.6B` continues a checkpoint on your own corpus with the trainer, sharding and checkpoints of every other Dew run.